# Lead Scoring — RF & HGB Hyperparameter Search

**Primary KPI** : `composite = AUC_test × 100 + decile_gap_pp`  (higher = better)

| Metric | What it measures |
|---|---|
| `AUC` | rank-ordering quality (0–1 → 0–100 after scaling) |
| `decile_gap_pp` | D10 SE − D1 SE (percentage points, 0–100) |
| `composite` | AUC×100 + gap_pp — single number to maximise |

**Sections** : Config → Data → Features → Split → Eval utils → Sweep (RF, HGB) → Leaderboard → Deep-dive → Correlation → Comparison

## 1 · Config & Imports

In [1]:
import re, time, warnings
from itertools import product as iterproduct

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.tree import export_text

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

/home/beastslayer/Desktop/projs/Genie-Stocks-Integration/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1b · Pipeline Config

In [2]:
# ══════════════════════════════════════════════════════════════
# PIPELINE CONFIG — single source of truth
# ══════════════════════════════════════════════════════════════

# ── Paths ──
DATA_PATH          = "../reports/test_scored.h5"
OPS_PATH           = "../reports/partner_ops_train_vector.csv"
COORDS_PATH        = "../reports/partner_coordinates.csv"

# ── Target ──
TARGET             = "installed_decision"

# ── Sweep ──
N_RANDOM_SAMPLES   = 120       # random combos per model type
TEST_SIZE          = 0.30
RANDOM_STATE       = 42
FILL_VALUE         = -999

# ── Temporal windows (auto-detected from ops CSV in §3, or override) ──
# TEMPORAL_WINDOWS will be populated after loading ops data.
# If you want to force specific windows, uncomment:
# TEMPORAL_WINDOWS = [30, 60, 365]

# ── Static ops columns (non-temporal, always included) ──
STATIC_OPS_COLS = [
    "queue_velocity", "plan_created_rate", "active_tickets",
    "nmbr_active_leads", "expected_daily_slots", "has_shock",
]

# ── Collinearity threshold ──
COLLINEAR_THRESHOLD = 0.7

# ── Plot theme ──
plt.rcParams.update({
    "figure.facecolor": "#0a0e17",
    "axes.facecolor":   "#111827",
    "axes.edgecolor":   "#1e293b",
    "axes.labelcolor":  "#9ca3af",
    "xtick.color":      "#6b7280",
    "ytick.color":      "#6b7280",
    "text.color":       "#e2e8f0",
    "grid.color":       "#1e293b",
    "font.family":      "monospace",
    "font.size":        10,
})

PALETTE = dict(
    amber="#f0b429", blue="#60a5fa", green="#10b981", red="#ef4444",
    purple="#a78bfa", cyan="#22d3ee", pink="#f472b6", gray="#4a5568",
)
AMBER, BLUE, GREEN, RED = PALETTE["amber"], PALETTE["blue"], PALETTE["green"], PALETTE["red"]
PURPLE, CYAN, PINK, GRAY = PALETTE["purple"], PALETTE["cyan"], PALETTE["pink"], PALETTE["gray"]

## 2 · Feature Registry

All features are defined **once** in `FEATURE_FAMILIES`.  
Temporal variants are generated automatically via `_temporal()` — adding a
new window means changing only `TEMPORAL_WINDOWS`.

**New**: `partner_geo` family — derived from DynamoDB partner coordinates
(`n_clusters`, `dist_to_partner_center_m`).

In [3]:
# ══════════════════════════════════════════════════════════════
# FEATURE FAMILIES — single source of truth
# ══════════════════════════════════════════════════════════════

def _temporal(names):
    """Return list of '{name}_{wd}d' for each name × window."""
    return [f"{n}_{wd}d" for wd in TEMPORAL_WINDOWS for n in names]


FEATURE_FAMILIES = {
    "spatial": {
        "base": [
            "predicted_field_hex",
            "predicted_field_hex_all_wmean",
            "predicted_field_hex_all_kswmean",
            "predicted_field_hex_all_min",
            "predicted_field_hex_all_max",
            "predicted_field_hex_all_std",
            "n_overlapping_hexes_field",
            "total_sources_all_hexes",
            "contested_field",
            "field_momentum",
        ],
        "temporal": ["predicted_field_hex"],
    },
    "hex": {
        "base": [
            "parent_se", "parent_total", "parent_color_numeric",
            "n_covering_partners", "parent_overlap", "install_velocity",
            "se_momentum",
        ],
        "temporal": ["weighted_se"],
        "temporal_suffix": "shrunk",
    },
    "hop": {
        "base": [
            "hop1_se_wmean", "hop2_se_wmean", "hop3_se_wmean",
            "hop1_se_std",   "hop2_se_std",   "hop3_se_std",
            "hop1_count",    "hop2_count",    "hop3_count",
            "se_gradient_1to3", "se_confirmed", "isolation_ratio",
        ],
        "temporal": [
            "hop1_se", "hop2_se", "hop3_se",
            "se_gradient_1to3", "se_confirmed",
        ],
        "temporal_suffix_map": {
            "hop1_se": "wmean", "hop2_se": "wmean", "hop3_se": "wmean",
        },
    },
    "contested": {
        "base": [
            "contested_area_km2", "contested_radius_m",
            "contested_se", "n_overlapping_partners",
        ],
        "temporal": ["contested_se"],
    },
    "boundary": {
        "base": [
            "dist_to_boundary_edge_point_hex",
            "dist_to_cluster_center_point_hex",
            "depth_score_point_hex",
            "mean_dist_to_edge_m", "mean_dist_to_center_m",
            "total_area_boundaries", "nmbr_overlap_clusters",
            "nearest_boundary_dist_m", "nmbr_boundaries_within_100m",
            "worst_depth_score", "any_near_edge", "is_solo_cluster",
        ],
    },
    "geometry": {
        "base": [
            "local_anisotropy", "local_density", "hull_area",
            "linearity_score", "spread_m",
            "dense_score", "gully_score", "sparse_score",
        ],
        "temporal": [
            "local_density", "dense_score", "sparse_score", "spread_m",
        ],
    },
    "lead": {
        "base": ["hard_density", "density_regime", "min_dist"],
    },
    "partner_geo": {
        "base": [
            "n_clusters",
            "partner_centroids_25m",
            "partner_centroids_50m",
            "partner_centroids_100m",
            "partner_centroids_200m",
            "nearest_centroid_dist_m",
        ],
    },
    "pipeline": {
        "base": [
            "install_pipeline_50m",
            "install_pipeline_100m",
            "install_pipeline_200m",
        ],
    },
    # ops family is populated dynamically after data load
}


def build_feature_list(families, ops_cols=None):
    """
    Expand FEATURE_FAMILIES into a flat list + family_map.

    Returns
    -------
    features : list[str]
    family_map : dict[str, str]   (feature_name → family_name)
    """
    features, family_map = [], {}

    for fam_name, spec in families.items():
        fam_feats = list(spec.get("base", []))

        # expand temporal features
        default_suffix = spec.get("temporal_suffix", None)
        suffix_map = spec.get("temporal_suffix_map", {})

        for name in spec.get("temporal", []):
            for wd in TEMPORAL_WINDOWS:
                sfx = suffix_map.get(name, default_suffix)
                col = f"{name}_{wd}d"
                if sfx:
                    col = f"{col}_{sfx}"
                fam_feats.append(col)

        for f in fam_feats:
            features.append(f)
            family_map[f] = fam_name

    # ops (dynamic)
    if ops_cols:
        for f in ops_cols:
            features.append(f)
            family_map[f] = "ops"

    return features, family_map

print("Feature families defined:",
      ", ".join(f"{k}({len(v.get('base',[]))}+{len(v.get('temporal',[]))}t)"
                for k, v in FEATURE_FAMILIES.items()))


Feature families defined: spatial(10+1t), hex(7+1t), hop(12+5t), contested(4+1t), boundary(12+0t), geometry(8+4t), lead(3+0t), partner_geo(6+0t), pipeline(3+0t)


## 3 · Data Load & Ops Merge

In [4]:
# ── Load scored leads ──
df = pd.read_hdf(DATA_PATH, "df")
print(f"Shape before ops merge: {df.shape}")

# ── Load ops (train-window only — leak-free) ──
df_ops_train = pd.read_csv(OPS_PATH)
print(f"Ops train: {df_ops_train.shape}")

# ── Auto-detect temporal windows from ops CSV ──
TEMPORAL_WINDOWS = sorted([
    int(c.replace("se_", "").replace("d", ""))
    for c in df_ops_train.columns
    if re.match(r"^se_\d+d$", c) and "delta" not in c
])
print(f"Temporal windows detected: {TEMPORAL_WINDOWS}")

Shape before ops merge: (18082, 204)
Ops train: (1475, 43)
Temporal windows detected: [30, 60, 365]


In [5]:
# ── Build ops column list dynamically from TEMPORAL_WINDOWS ──
smallest_wd = min(TEMPORAL_WINDOWS)

# Temporal ops metrics per window
OPS_TEMPORAL_METRICS = ["se", "decline_rate", "median_response_min"]
OPS_DELTA_METRICS    = ["se_delta", "decline_rate_delta", "response_delta"]

rf_ops_cols = ["partner_id"]
for wd in TEMPORAL_WINDOWS:
    rf_ops_cols += [f"{m}_{wd}d" for m in OPS_TEMPORAL_METRICS]
    if wd != smallest_wd:
        rf_ops_cols += [f"{m}_{smallest_wd}_{wd}" for m in OPS_DELTA_METRICS]

rf_ops_cols += STATIC_OPS_COLS
rf_ops_cols = [c for c in rf_ops_cols if c in df_ops_train.columns]

# ── Merge ──
df["partner_id"] = df["partner_id"].astype(str)
df_ops_train["partner_id"] = df_ops_train["partner_id"].astype(str)
df = df.merge(df_ops_train[rf_ops_cols], on="partner_id", how="left", suffixes=("", "_ops"))
df = df.loc[:, ~df.columns.str.endswith("_ops")]
print(f"Shape after ops merge: {df.shape}")

_ops_feature_cols = [c for c in rf_ops_cols if c != "partner_id"]
print(f"Ops columns merged: {len(_ops_feature_cols)}")

Shape after ops merge: (18082, 223)
Ops columns merged: 19


### 3b · Partner Coordinates Merge

Loads partner cluster centroids from the coordinates CSV (separate from ops).
Uses **BallTree** on each partner's centroids to compute per-lead radius features:

- `n_clusters` — total hex clusters the partner operates
- `partner_centroids_25m / 50m / 100m / 200m` — how many centroids fall within each radius of the lead
- `nearest_centroid_dist_m` — distance to the closest centroid

These capture whether the lead sits in the core vs fringe of a partner's territory.


In [6]:
import json as _json
from sklearn.neighbors import BallTree as _BallTree

CENTROID_RADII_M = [25, 50, 100, 200]
EARTH_R = 6_371_000  # metres

# ── Load coordinates (separate artifact, not in ops vector) ──
import os
if os.path.exists(COORDS_PATH):
    df_coords_raw = pd.read_csv(COORDS_PATH)
    print(f"Partner coordinates loaded: {df_coords_raw.shape}")

    # ── Parse JSON → per-partner list of centroid arrays ──
    _partner_centroids = {}   # partner_id → np.array([[lat_rad, lon_rad], ...])
    _partner_n_clusters = {}

    for _, row in df_coords_raw.iterrows():
        pid = str(row["partner_id"])
        raw = row.get("coordinates", None)

        if pd.isna(raw) or not raw:
            _partner_n_clusters[pid] = 0
            continue
        try:
            clusters = _json.loads(raw) if isinstance(raw, str) else raw
            centroids = [c["centroid"] for c in clusters
                         if "centroid" in c and c["centroid"] and c["centroid"][0] is not None]
            if centroids:
                _partner_centroids[pid] = np.radians(np.array(centroids, dtype=float))
                _partner_n_clusters[pid] = len(centroids)
            else:
                _partner_n_clusters[pid] = 0
        except Exception:
            _partner_n_clusters[pid] = 0

    print(f"  Partners with centroids: {len(_partner_centroids)}")
    print(f"  Median clusters/partner: {np.median(list(_partner_n_clusters.values())):.0f}")

    # ── Pre-build a BallTree per partner (only for partners with ≥1 centroid) ──
    _partner_trees = {}
    for pid, coords_rad in _partner_centroids.items():
        if len(coords_rad) >= 1:
            _partner_trees[pid] = _BallTree(coords_rad, metric="haversine")

    # ── Compute per-lead features ──
    df["partner_id"] = df["partner_id"].astype(str)
    lead_coords_rad = np.radians(df[["latitude", "longitude"]].values)

    n_clusters_arr = np.zeros(len(df), dtype=int)
    nearest_dist_arr = np.full(len(df), np.nan)
    radius_counts = {r: np.zeros(len(df), dtype=int) for r in CENTROID_RADII_M}

    for idx in range(len(df)):
        pid = df["partner_id"].iloc[idx]
        n_clusters_arr[idx] = _partner_n_clusters.get(pid, 0)

        tree = _partner_trees.get(pid)
        if tree is None:
            continue

        pt = lead_coords_rad[idx].reshape(1, -1)

        # Nearest centroid
        dist_rad, _ = tree.query(pt, k=1)
        nearest_dist_arr[idx] = dist_rad[0, 0] * EARTH_R

        # Radius counts
        for r_m in CENTROID_RADII_M:
            r_rad = r_m / EARTH_R
            count = tree.query_radius(pt, r=r_rad, count_only=True)[0]
            radius_counts[r_m][idx] = count

    # ── Assign to df ──
    df["n_clusters"] = n_clusters_arr
    df["nearest_centroid_dist_m"] = nearest_dist_arr
    for r_m in CENTROID_RADII_M:
        df[f"partner_centroids_{r_m}m"] = radius_counts[r_m]

    # ── Report ──
    print(f"\n  Feature summary:")
    print(f"    n_clusters         — median: {df['n_clusters'].median():.0f}, max: {df['n_clusters'].max():.0f}")
    print(f"    nearest_centroid   — median: {df['nearest_centroid_dist_m'].median():.0f}m, "
          f"p90: {df['nearest_centroid_dist_m'].quantile(0.9):.0f}m")
    for r_m in CENTROID_RADII_M:
        col = f"partner_centroids_{r_m}m"
        n_pos = (df[col] > 0).sum()
        print(f"    {col:25s} — leads with ≥1: {n_pos:,} ({n_pos/len(df)*100:.1f}%), "
              f"mean: {df[col].mean():.2f}")
else:
    print(f"WARNING: {COORDS_PATH} not found — partner_geo features will be NaN")
    df["n_clusters"] = np.nan
    df["nearest_centroid_dist_m"] = np.nan
    for r_m in CENTROID_RADII_M:
        df[f"partner_centroids_{r_m}m"] = np.nan

print(f"\nShape after coordinates merge: {df.shape}")



Shape after coordinates merge: (18082, 229)


### 3c · Install Pipeline Density (Leak-Free)

For each test lead at time `t0`, count how many **other** leads from the
**same partner** had their `decision_time` within the **backward** window
`[t0 − 2 days, t0]`, at radii of 50 m, 100 m, and 200 m.

**No leakage:**
- Uses ALL leads as source — never filters by `installed_decision` (the target).
- Backward window only — never peeks at leads arriving after `t0`.
- Excludes the lead itself (by `mobile`).

In [7]:
# ── Install Pipeline Density config ──
PIPELINE_RADII_M     = [50, 100, 200]
PIPELINE_WINDOW_DAYS = 2          # backward window
EARTH_R_PIPELINE     = 6_371_000  # metres

# Source = ALL leads with valid coords + time (NO outcome filter)
pipe_src = df[
    df["latitude"].notna()
    & df["longitude"].notna()
    & pd.to_datetime(df["decision_time"], errors="coerce").notna()
].copy()
pipe_src["partner_id"]   = pipe_src["partner_id"].astype(str)
pipe_src["decision_time"] = pd.to_datetime(pipe_src["decision_time"])
pipe_src["mobile"]       = pipe_src["mobile"].astype(str)

print(f"Pipeline source (ALL leads, no outcome filter): {len(pipe_src):,}")
print(f"Unique partners in source: {pipe_src['partner_id'].nunique():,}")

Pipeline source (ALL leads, no outcome filter): 18,082
Unique partners in source: 1,079


In [8]:
from sklearn.neighbors import BallTree as _PipeBT

# ── Build one BallTree per partner ──
df["partner_id"]   = df["partner_id"].astype(str)
df["decision_time"] = pd.to_datetime(df["decision_time"], errors="coerce")
df["mobile"]       = df["mobile"].astype(str)

_pipe_by_partner = {}
for pid, grp in pipe_src.groupby("partner_id"):
    coords_rad = np.radians(grp[["latitude", "longitude"]].values)
    times      = grp["decision_time"].values.astype("datetime64[ns]")
    mobiles    = grp["mobile"].values.astype(str)
    tree       = _PipeBT(coords_rad, metric="haversine")
    _pipe_by_partner[pid] = (tree, coords_rad, times, mobiles)

print(f"Built BallTrees for {len(_pipe_by_partner)} partners")

Built BallTrees for 1079 partners


In [9]:
# ── Compute pipeline counts per lead (backward, self-excluded) ──
n = len(df)
pipeline_counts = {
    f"install_pipeline_{r}m": np.zeros(n, dtype=int)
    for r in PIPELINE_RADII_M
}
radii_rad  = [r / EARTH_R_PIPELINE for r in PIPELINE_RADII_M]
max_r_rad  = max(radii_rad)
window_ns  = np.timedelta64(PIPELINE_WINDOW_DAYS, "D")

lead_pids    = df["partner_id"].values
lead_times   = df["decision_time"].values.astype("datetime64[ns]")
lead_mobiles = df["mobile"].values.astype(str)
lead_coords  = np.radians(df[["latitude", "longitude"]].values)

In [10]:
for i in range(n):
    pid = lead_pids[i]
    t0  = lead_times[i]
    mob = lead_mobiles[i]

    if np.isnat(t0) or pid not in _pipe_by_partner:
        continue

    tree, _, pipe_times, pipe_mobiles = _pipe_by_partner[pid]
    pt = lead_coords[i].reshape(1, -1)

    # Spatial query at max radius
    idxs, dists = tree.query_radius(
        pt, r=max_r_rad, return_distance=True
    )
    idxs, dists = idxs[0], dists[0]
    if len(idxs) == 0:
        continue

    # Exclude SELF (same mobile)
    not_self = pipe_mobiles[idxs] != mob
    idxs  = idxs[not_self]
    dists = dists[not_self]
    if len(idxs) == 0:
        continue

    # BACKWARD time window: [t0 - 2d, t0]  (no future peek)
    neigh_times = pipe_times[idxs]
    time_ok = (neigh_times >= t0 - window_ns) & (neigh_times <= t0)
    valid_dists = dists[time_ok]

    # Count per radius
    for r, r_rad in zip(PIPELINE_RADII_M, radii_rad):
        col = f"install_pipeline_{r}m"
        pipeline_counts[col][i] = int((valid_dists <= r_rad).sum())

In [11]:
# ── Assign to df & report ──
for r in PIPELINE_RADII_M:
    col = f"install_pipeline_{r}m"
    df[col] = pipeline_counts[col]

print("Install Pipeline Density features (leak-free):")
for r in PIPELINE_RADII_M:
    col = f"install_pipeline_{r}m"
    n_pos = (df[col] > 0).sum()
    med_nz = df.loc[df[col] > 0, col].median() if n_pos > 0 else 0
    print(f"  {col:28s}  leads with ≥1: {n_pos:>6,}"
          f" ({n_pos/len(df)*100:5.1f}%)  median|nz: {med_nz:.0f}")

print(f"\nShape after pipeline merge: {df.shape}")

Install Pipeline Density features (leak-free):
  install_pipeline_50m          leads with ≥1:  1,934 ( 10.7%)  median|nz: 1
  install_pipeline_100m         leads with ≥1:  3,028 ( 16.7%)  median|nz: 1
  install_pipeline_200m         leads with ≥1:  5,322 ( 29.4%)  median|nz: 1

Shape after pipeline merge: (18082, 232)


## 4 · Feature Selection

In [12]:
# ── Build full feature list from registry ──
FEATURES, family_map = build_feature_list(FEATURE_FAMILIES, ops_cols=_ops_feature_cols)

available = [f for f in FEATURES if f in df.columns]
missing   = [f for f in FEATURES if f not in df.columns]

print(f"Using {len(available)}/{len(FEATURES)} features  |  missing: {len(missing)}")
if missing:
    for m in missing:
        print(f"  ✗ {m}")

# Show breakdown by family
from collections import Counter
fam_counts = Counter(family_map[f] for f in available)
for fam, n in sorted(fam_counts.items(), key=lambda x: -x[1]):
    print(f"  {fam:12s}  {n:3d} features")

Using 120/120 features  |  missing: 0
  hop            27 features
  geometry       20 features
  ops            19 features
  spatial        13 features
  boundary       12 features
  hex            10 features
  contested       7 features
  partner_geo     6 features
  lead            3 features
  pipeline        3 features


## 5 · Correlation & Collinearity Analysis

Run **before** model training to understand feature–target relationships
and identify redundant features upfront.

In [13]:
# ── Feature vs target correlation ──
feat_df = df[available + [TARGET]].copy()
corr_target = feat_df.corr()[TARGET].drop(TARGET).sort_values(ascending=False).fillna(0)

print("Correlation with installed_decision:\n")
for feat, val in corr_target.items():
    bar = "█" * int(abs(val) * 200)
    sign = "+" if val >= 0 else "-"
    print(f"  {sign}{abs(val):.4f}  {feat:40s}  {bar}")

# ── High collinearity pairs ──
corr_matrix = df[available].corr()
high_corr = []
for i in range(len(available)):
    for j in range(i + 1, len(available)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > COLLINEAR_THRESHOLD:
            high_corr.append((available[i], available[j], round(r, 3)))

high_corr.sort(key=lambda x: -abs(x[2]))
print(f"\nHighly correlated pairs (|r| > {COLLINEAR_THRESHOLD}): {len(high_corr)}\n")
for a, b, r in high_corr[:30]:
    print(f"  {r:+.3f}  {a}  <->  {b}")

Correlation with installed_decision:

  +0.2032  se_60d                                    ████████████████████████████████████████
  +0.2015  se_30d                                    ████████████████████████████████████████
  +0.1856  contested_se_60d                          █████████████████████████████████████
  +0.1698  contested_se_365d                         █████████████████████████████████
  +0.1609  se_365d                                   ████████████████████████████████
  +0.1536  predicted_field_hex_60d                   ██████████████████████████████
  +0.1454  contested_se_30d                          █████████████████████████████
  +0.1450  predicted_field_hex_30d                   ████████████████████████████
  +0.1381  predicted_field_hex                       ███████████████████████████
  +0.1381  predicted_field_hex_all_kswmean           ███████████████████████████
  +0.1381  predicted_field_hex_365d                  ███████████████████████████
  +0.1370  se_delt

## 6 · Prep & Train/Test Split

In [14]:
X = df[available].copy()
y = df[TARGET].copy()

mask = y.notna()
X, y = X[mask], y[mask]
X = X.fillna(FILL_VALUE)

print(f"Samples: {len(X)}  |  Features: {X.shape[1]}  |  Install rate: {y.mean():.4f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y.astype(int),
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"Train SE: {y_train.mean():.4f}  |  Test SE: {y_test.mean():.4f}")

Samples: 18082  |  Features: 120  |  Install rate: 0.5794
Train: 12657  |  Test: 5425
Train SE: 0.5794  |  Test SE: 0.5794


## 7 · Evaluation Utilities

**Composite KPI** = `AUC_test × 100 + gap_pp`

- AUC ∈ [0.5, 1.0] → scaled to [50, 100]
- gap_pp = (D10_SE − D1_SE) × 100 → typically 0–60
- composite ≈ 50–160 range; higher is better

In [15]:
def eval_model(model, X_tr, y_tr, X_te, y_te):
    """Evaluate a fitted model. Returns dict with all metrics."""
    p_train = model.predict(X_tr)
    p_test  = model.predict(X_te)

    auc_train = roc_auc_score(y_tr, p_train)
    auc_test  = roc_auc_score(y_te, p_test)

    tbl = _decile_table(y_te.values, p_test)

    d1_se  = tbl["se"].iloc[0]
    d10_se = tbl["se"].iloc[-1]
    gap_pp = (d10_se - d1_se) * 100
    mono   = bool((tbl["se"].diff().dropna() >= 0).all())

    composite = auc_test * 100 + gap_pp

    return dict(
        auc_train=round(auc_train, 4),
        auc_test=round(auc_test, 4),
        gap_pp=round(gap_pp, 2),
        d1_se=round(d1_se, 4),
        d10_se=round(d10_se, 4),
        mono=mono,
        composite=round(composite, 2),
        decile_tbl=tbl,
    )


def _decile_table(y_true, y_pred, q=10):
    """Decile table: install rate per prediction quantile bin."""
    edf = pd.DataFrame({"y": y_true, "p": y_pred})
    edf["decile"] = pd.qcut(edf["p"], q=q, labels=False, duplicates="drop") + 1
    tbl = edf.groupby("decile").agg(n=("y", "count"), inst=("y", "sum"))
    tbl["se"] = tbl["inst"] / tbl["n"]
    return tbl


def score_column(df, score_col, target=TARGET):
    """Quick AUC + decile gap for any pre-computed score column."""
    sub = df[df[target].notna()]
    auc_val = roc_auc_score(sub[target], sub[score_col])
    tbl = _decile_table(sub[target].values, sub[score_col].values)
    gap = (tbl["se"].iloc[-1] - tbl["se"].iloc[0]) * 100
    mono = bool((tbl["se"].diff().dropna() >= 0).all())
    comp = auc_val * 100 + gap
    return dict(auc=round(auc_val, 4), gap_pp=round(gap, 2), mono=mono, composite=round(comp, 2))


def print_leaderboard(results_df, title, top_n=15):
    """Print a compact leaderboard sorted by composite KPI."""
    df_s = (results_df
            .sort_values("composite", ascending=False)
            .head(top_n)
            .reset_index(drop=True))
    print(f"\n{'='*90}")
    print(f"  {title}  (top {top_n} by composite = AUC×100 + gap_pp)")
    print(f"{'='*90}")
    cols = [c for c in df_s.columns if c != "decile_tbl"]
    print(df_s[cols].to_string(index=True))
    return df_s

## 8 · Hyperparameter Sweep

A single `run_sweep()` function handles both RF and HGB to avoid
duplicated sweep logic.

In [16]:
def run_sweep(model_class, param_grid, fixed_params=None, label="model"):
    """
    Random hyperparameter sweep.

    Parameters
    ----------
    model_class : sklearn estimator class
    param_grid  : dict[str, list]  — values to sweep
    fixed_params: dict — always passed to the constructor
    label       : str  — display name (RF / HGB)

    Returns
    -------
    results_df  : pd.DataFrame   — one row per experiment
    """
    fixed_params = fixed_params or {}
    keys = list(param_grid.keys())
    all_combos = list(iterproduct(*param_grid.values()))

    np.random.seed(RANDOM_STATE)
    n_sample = min(N_RANDOM_SAMPLES, len(all_combos))
    idx = np.random.choice(len(all_combos), size=n_sample, replace=False)
    sampled = [all_combos[i] for i in idx]

    print(f"{label} grid: {len(all_combos)} total combos → sampling {n_sample}")

    rows = []
    best_composite = -np.inf

    for combo in tqdm(sampled, desc=f"{label} sweep"):
        params = dict(zip(keys, combo))
        model = model_class(**params, **fixed_params)
        model.fit(X_train, y_train)
        ev = eval_model(model, X_train, y_train, X_test, y_test)

        row = {"model": label, **params, **{k: ev[k] for k in
               ["auc_train", "auc_test", "gap_pp", "mono", "composite"]}}

        # track n_iter for early-stopped models
        if hasattr(model, "n_iter_"):
            row["n_iter"] = model.n_iter_

        rows.append(row)
        if ev["composite"] > best_composite:
            best_composite = ev["composite"]
            print(f"  ★ new best | AUC={ev['auc_test']:.4f}  gap_pp={ev['gap_pp']:.2f}  composite={best_composite:.2f}")

    print(f"\n{label} sweep done — best composite: {best_composite:.2f}")
    return pd.DataFrame(rows)

### 8a · Random Forest Sweep

In [17]:
RF_GRID = {
    "n_estimators":     [200, 300, 500],
    "max_depth":        [6, 8, 12, 16, None],
    "min_samples_leaf": [20, 50, 100, 200],
    "max_features":     ["sqrt", 0.3, 0.5],
}

RF_FIXED = dict(random_state=RANDOM_STATE, n_jobs=-1)

rf_df = run_sweep(RandomForestRegressor, RF_GRID, RF_FIXED, label="RF")
print_leaderboard(rf_df, "RANDOM FOREST LEADERBOARD");

RF grid: 180 total combos → sampling 120


RF sweep:   1%|          | 1/120 [00:00<01:52,  1.06it/s]

  ★ new best | AUC=0.6870  gap_pp=61.69  composite=130.40


RF sweep:   2%|▏         | 2/120 [00:01<01:15,  1.57it/s]

  ★ new best | AUC=0.6878  gap_pp=62.62  composite=131.39


...

  ★ new best | AUC=0.6995  gap_pp=69.43  composite=139.38


RF sweep:  83%|████████▎ | 100/120 [02:39<00:17,  1.16it/s]

  ★ new best | AUC=0.6993  gap_pp=69.98  composite=139.91


RF sweep: 100%|██████████| 120/120 [03:15<00:00,  1.63s/it]


RF sweep done — best composite: 139.91

  RANDOM FOREST LEADERBOARD  (top 15 by composite = AUC×100 + gap_pp)
   model  n_estimators  max_depth  min_samples_leaf max_features  auc_train  auc_test  gap_pp   mono  composite
0     RF           300        NaN                20         sqrt     0.8975    0.6993   69.98   True     139.91
1     RF           500        NaN                20         sqrt     0.8978    0.7004   69.61  False     139.65
2     RF           500       12.0                20         sqrt     0.8736    0.6995   69.43  False     139.38
3     RF           200       16.0                20         sqrt     0.8941    0.6994   69.06  False     139.00
4     RF           500       16.0                20         sqrt     0.8949    0.6999   68.88   True     138.86
5     RF           200       12.0                20         sqrt     0.8727    0.6972   69.06   True     138.78
6     RF           300       12.0                20         sqrt     0.8735    0.6986   68.88  False     

### 8b · HistGradientBoosting Sweep

In [18]:
HGB_GRID = {
    "learning_rate":     [0.02, 0.04, 0.07, 0.10],
    "max_depth":         [3, 4, 6, 8],
    "min_samples_leaf":  [30, 50, 100, 200],
    "l2_regularization": [0.5, 2.0, 5.0, 10.0],
    "max_leaf_nodes":    [15, 31, 63],
}

HGB_FIXED = dict(
    loss="squared_error",
    max_iter=1000,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=RANDOM_STATE,
)

hgb_df = run_sweep(HistGradientBoostingRegressor, HGB_GRID, HGB_FIXED, label="HGB")
print_leaderboard(hgb_df, "HGB LEADERBOARD");

HGB grid: 768 total combos → sampling 120


HGB sweep:   1%|          | 1/120 [00:00<00:27,  4.34it/s]

  ★ new best | AUC=0.6918  gap_pp=62.43  composite=131.61


HGB sweep:   2%|▏         | 2/120 [00:00<00:40,  2.91it/s]

  ★ new best | AUC=0.6920  gap_pp=65.93  composite=135.13


HGB sweep:  52%|█████▏    | 62/120 [00:20<00:17,  3.40it/s]

  ★ new best | AUC=0.6939  gap_pp=67.03  composite=136.43


HGB sweep: 100%|██████████| 120/120 [00:41<00:00,  2.86it/s]


HGB sweep done — best composite: 136.43

  HGB LEADERBOARD  (top 15 by composite = AUC×100 + gap_pp)
   model  learning_rate  max_depth  min_samples_leaf  l2_regularization  max_leaf_nodes  auc_train  auc_test  gap_pp   mono  composite  n_iter
0    HGB           0.10          8               200                5.0              63     0.8612    0.6939   67.03   True     136.43      87
1    HGB           0.04          6               200                0.5              15     0.8028    0.6920   65.93  False     135.13     230
2    HGB           0.04          8               100                0.5              31     0.8481    0.6947   65.56  False     135.03     156
3    HGB           0.07          8               200                2.0              31     0.8235    0.6955   65.38   True     134.93      82
4    HGB           0.07          6               100                5.0              31     0.8334    0.6945   65.19   True     134.65     101
5    HGB           0.04          8      

## 9 · Combined Leaderboard

In [19]:
combined = pd.concat([rf_df, hgb_df], ignore_index=True)
combined_sorted = combined.sort_values("composite", ascending=False).reset_index(drop=True)

# ── Pretty-print top 20 ──
display_cols = ["model", "auc_test", "gap_pp", "composite", "mono"]
param_cols = [c for c in combined_sorted.columns if c not in display_cols + ["auc_train", "decile_tbl", "n_iter"]]
print(combined_sorted.head(20)[display_cols + param_cols].to_string())

# ── Summary per model type ──
for m in ["RF", "HGB"]:
    sub = combined_sorted[combined_sorted["model"] == m]
    print(f"\n{m} — best composite: {sub['composite'].max():.2f}  |  "
          f"median: {sub['composite'].median():.2f}  |  "
          f"best AUC: {sub['auc_test'].max():.4f}  |  "
          f"best gap_pp: {sub['gap_pp'].max():.2f}")

   model  auc_test  gap_pp  composite   mono  n_estimators  max_depth  min_samples_leaf max_features  learning_rate  l2_regularization  max_leaf_nodes
0     RF    0.6993   69.98     139.91   True         300.0        NaN                20         sqrt            NaN                NaN             NaN
1     RF    0.7004   69.61     139.65  False         500.0        NaN                20         sqrt            NaN                NaN             NaN
2     RF    0.6995   69.43     139.38  False         500.0       12.0                20         sqrt            NaN                NaN             NaN
3     RF    0.6994   69.06     139.00  False         200.0       16.0                20         sqrt            NaN                NaN             NaN
4     RF    0.6999   68.88     138.86   True         500.0       16.0                20         sqrt            NaN                NaN             NaN
5     RF    0.6972   69.06     138.78   True         200.0       12.0                20       

## 10 · Best-Model Deep Dive

In [20]:
def refit_best(results_df, model_class, fixed_params=None):
    """Refit the best config from a sweep DataFrame."""
    fixed_params = fixed_params or {}
    best_row = results_df.sort_values("composite", ascending=False).iloc[0]

    # Extract only the params that the model constructor accepts
    skip = {"model", "auc_train", "auc_test", "gap_pp", "mono", "composite", "n_iter", "decile_tbl"}
    params = {k: (int(v) if isinstance(v, (np.integer, float)) and v == int(v) and k != "max_features"
                  else (None if pd.isna(v) else v))
              for k, v in best_row.items() if k not in skip}

    model = model_class(**params, **fixed_params)
    model.fit(X_train, y_train)
    ev = eval_model(model, X_train, y_train, X_test, y_test)
    return model, ev, best_row


best_rf, ev_rf, _   = refit_best(rf_df, RandomForestRegressor, RF_FIXED)
best_hgb, ev_hgb, _ = refit_best(hgb_df, HistGradientBoostingRegressor, HGB_FIXED)

for name, ev in [("RF", ev_rf), ("HGB", ev_hgb)]:
    print(f"Best {name:3s} — AUC: {ev['auc_test']:.4f}  gap_pp: {ev['gap_pp']:.2f}  "
          f"composite: {ev['composite']:.2f}  mono: {ev['mono']}")

# Pick overall winner
if ev_hgb["composite"] >= ev_rf["composite"]:
    best_model, ev_best, best_name = best_hgb, ev_hgb, "HGB"
else:
    best_model, ev_best, best_name = best_rf, ev_rf, "RF"
print(f"\n>>> Winner: {best_name}  composite={ev_best['composite']:.2f}")

ValueError: cannot convert float NaN to integer

### 10a · Decile Curve & ROC

In [ ]:
p_rf  = best_rf.predict(X_test)
p_hgb = best_hgb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Decile curves ──
ax = axes[0]
for preds, label, color, marker in [
    (p_rf,  "RF",  BLUE,  "o"),
    (p_hgb, "HGB", AMBER, "s"),
]:
    tbl = _decile_table(y_test.values, preds)
    d = list(range(1, len(tbl) + 1))
    ax.plot(d, tbl["se"].values, f"{marker}-", color=color, linewidth=2,
            markersize=6, label=label)
    for j, se in enumerate(tbl["se"].values):
        ax.annotate(f"{se:.3f}", (j+1, se), fontsize=6, ha="center", va="bottom",
                    xytext=(0, 5), textcoords="offset points", color=color)

ax.axhline(y_test.mean(), color=GRAY, linestyle=":", linewidth=0.8, label="base rate")
ax.set(xlabel="Decile", ylabel="SE (Install Rate)")
ax.set_title("DECILE CALIBRATION (test)", fontsize=12, fontweight="bold", color=AMBER, pad=10)
ax.set_xticks(range(1, 11))
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(alpha=0.2)

# ── ROC ──
ax = axes[1]
for preds, label, color, ls in [
    (p_rf,  "RF",  BLUE,  "--"),
    (p_hgb, "HGB", AMBER, "-"),
]:
    fpr, tpr, _ = roc_curve(y_test, preds)
    a = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, linestyle=ls,
            label=f"{label} (AUC={a:.4f})")

ax.plot([0, 1], [0, 1], ":", color=GRAY, linewidth=0.8)
ax.set(xlabel="FPR", ylabel="TPR")
ax.set_title("ROC CURVE (test)", fontsize=12, fontweight="bold", color=AMBER, pad=10)
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### 10b · Feature Importance (best RF)

In [ ]:
# ── Family → color lookup (derived from registry) ──
FAM_COLORS = {
    "spatial": AMBER, "hex": BLUE, "hop": PURPLE, "contested": PINK,
    "boundary": CYAN, "geometry": GREEN, "lead": RED, "ops": "#ff6b6b",
    "partner_geo": "#facc15",
    "pipeline": "#fb923c",
}

imp = pd.Series(best_rf.feature_importances_, index=available).sort_values(ascending=True)
colors = [FAM_COLORS.get(family_map.get(f, ""), GRAY) for f in imp.index]

fig, ax = plt.subplots(figsize=(12, max(10, len(imp) * 0.28)))
ax.barh(range(len(imp)), imp.values, color=colors, height=0.7, edgecolor="none")
ax.set_yticks(range(len(imp)))
ax.set_yticklabels(imp.index, fontsize=8)
ax.set_xlabel("Gini Importance")
ax.set_title("FEATURE IMPORTANCE BY FAMILY — best RF",
             fontsize=14, fontweight="bold", color=AMBER, pad=12)
ax.grid(axis="x", alpha=0.3)

patches = [mpatches.Patch(color=c, label=k) for k, c in FAM_COLORS.items()]
ax.legend(handles=patches, loc="lower right", fontsize=8, framealpha=0.3, edgecolor="#1e293b")
plt.tight_layout()
plt.show()


### 10c · Single Tree (depth=4 excerpt)

In [ ]:
single_tree = best_rf.estimators_[0]
print(export_text(single_tree, feature_names=available, max_depth=4))

### 10d · Importance vs Correlation Scatter

In [ ]:
imp_desc = pd.Series(best_rf.feature_importances_, index=available)

fig, ax = plt.subplots(figsize=(10, 8))
for feat in imp_desc.index:
    c = FAM_COLORS.get(family_map.get(feat, ""), GRAY)
    ax.scatter(corr_target.get(feat, 0), imp_desc[feat],
               c=c, s=60, alpha=0.85, edgecolor="white", linewidth=0.3, zorder=3)

for feat in imp_desc.nlargest(10).index:
    if feat in corr_target.index:
        ax.annotate(feat, (corr_target[feat], imp_desc[feat]),
                    fontsize=7, color="#9ca3af", ha="left",
                    xytext=(5, 5), textcoords="offset points")

ax.axvline(0, color=GRAY, linestyle="--", linewidth=0.8, alpha=0.5)
ax.set(xlabel="Pearson r with target", ylabel="RF Importance")
ax.set_title("IMPORTANCE vs CORRELATION", fontsize=11, fontweight="bold", color=AMBER, pad=12)
ax.grid(alpha=0.2)

patches = [mpatches.Patch(color=c, label=k) for k, c in FAM_COLORS.items()]
ax.legend(handles=patches, loc="upper left", fontsize=8, framealpha=0.3, edgecolor="#1e293b")
plt.tight_layout()
plt.show()

## 11 · Comparison with Existing Scores

In [ ]:
# Score the full dataset with both best models
df["rf_score"]  = best_rf.predict(df[available].fillna(FILL_VALUE))
df["hgb_score"] = best_hgb.predict(df[available].fillna(FILL_VALUE))

COMPARE_COLS = ["rf_score", "hgb_score", "composite_score", "spatial_shrunk"]

print(f"{'Score':25s}  {'AUC':>7s}  {'gap_pp':>7s}  {'composite':>10s}  {'mono':>5s}")
print("-" * 62)
for col in COMPARE_COLS:
    if col in df.columns:
        m = score_column(df, col)
        print(f"{col:25s}  {m['auc']:7.4f}  {m['gap_pp']:7.2f}  "
              f"{m['composite']:10.2f}  {str(m['mono']):>5s}")